# Overall analysis

In [ ]:
import pandas as pd

import numpy as np
import matplotlib.pyplot as plt

# compare Gem TL

In [ ]:
# data_Gem = pd.read_csv('./GEMINI-32nm/dse_log/eyeriss_resnet50_B64/result.csv').astype(float)
# data_Gem_TL = pd.read_csv('./GEMINI-TL-real/dse_log/G_resnet50/result.csv').astype(float)
# data_TLP = pd.read_csv('./GEMINI-TL/dse_log/eyeriss_resnet50_B64/result.csv').astype(float)
# data_TLP_TL = pd.read_csv('./GEMINI-TL-real/dse_log/TL_resnet50/result.csv').astype(float)

data_Gem = pd.read_csv('./GEMINI-32nm/dse_log/eyeriss_transformer_B64/result.csv').astype(float)
data_Gem_TL = pd.read_csv('./GEMINI-TL-real/dse_log/G_transformer/result.csv').astype(float)
data_TLP = pd.read_csv('./GEMINI-TL/dse_log/eyeriss_transformer_B64/result.csv').astype(float)
data_TLP_TL = pd.read_csv('./GEMINI-TL-real/dse_log/TL_transformer/result.csv').astype(float)

In [ ]:
mapping_gem = np.zeros(data_Gem.values.shape[0], dtype=int)
for data_idx in range(len(mapping_gem)):
    mapping_rst = data_Gem.values[data_idx,:16] == data_Gem_TL.values[:,:16]
    mapping_rst = np.sum(mapping_rst, axis=1) == 16
    try:
        mapping_gem[data_idx] = np.where(mapping_rst)[0][0]
    except:
        mapping_gem[data_idx] = -1

data_Gem_val = data_Gem.values[mapping_gem>-1]
data_Gem_TL_val = data_Gem_TL.values[mapping_gem[mapping_gem>-1]]

valid_mapping = data_Gem_TL_val[:,19] < 1e23
data_Gem_val = data_Gem_val[valid_mapping]
data_Gem_TL_val  = data_Gem_TL_val[valid_mapping]


mapping_gem = np.zeros(data_TLP.values.shape[0], dtype=int)
for data_idx in range(len(mapping_gem)):
    mapping_rst = data_TLP.values[data_idx,:16] == data_TLP_TL.values[:,:16]
    mapping_rst = np.sum(mapping_rst, axis=1) == 16
    try:
        mapping_gem[data_idx] = np.where(mapping_rst)[0][0]
    except:
        mapping_gem[data_idx] = -1

data_TLP_val = data_TLP.values[mapping_gem>-1]
data_TLP_TL_val = data_TLP_TL.values[mapping_gem[mapping_gem>-1]]

valid_mapping = data_TLP_TL_val[:,19] < 1e23
data_TLP_val = data_TLP_val[valid_mapping]
data_TLP_TL_val  = data_TLP_TL_val[valid_mapping]

In [ ]:
targets = [0, 1, 2]
factors = [1, 1, 2]

matrice_Gem, matrice_Gem_TL = np.ones(data_Gem_val.shape[0]), np.ones(data_Gem_val.shape[0])
for i in range(len(factors)):
    matrice_Gem = matrice_Gem * (data_Gem_val[:,targets[i]+16] ** factors[i])
    matrice_Gem_TL = matrice_Gem_TL * (data_Gem_TL_val[:,targets[i]+16] ** factors[i])

matrice_TLP, matrice_TLP_TL = np.ones(data_TLP_val.shape[0]), np.ones(data_TLP_val.shape[0])
for i in range(len(factors)):
    matrice_TLP = matrice_TLP * (data_TLP_val[:,targets[i]+16] ** factors[i])
    matrice_TLP_TL = matrice_TLP_TL * (data_TLP_TL_val[:,targets[i]+16] ** factors[i])

In [ ]:
bins = np.arange(-1, 1.1, 0.1)

err_Gem = (matrice_Gem-matrice_Gem_TL)/matrice_Gem_TL
err_TLP = (matrice_TLP-matrice_TLP_TL)/matrice_TLP_TL
plt.hist(err_Gem, bins=bins, alpha=0.7, label='Gemini', color='tab:blue')
plt.hist(err_TLP, bins=bins, alpha=0.7, label='Predictor', color='tab:orange')
plt.legend()
plt.ylabel("Data Count")
plt.xlabel("Normalized Error Rate")
plt.title(r'$M^{:} \times E^{:} \times T^{:}$ on ResNet'.format(factors[0], factors[1], factors[2]))
plt.show()


In [ ]:
select_row = np.argsort(matrice_Gem)
plt.scatter(matrice_Gem[select_row[:40]]/np.max(matrice_Gem[select_row[:40]]), matrice_Gem_TL[select_row[:40]]/np.max(matrice_Gem_TL[select_row[:40]]), color="tab:blue", label="Gemini")
# plt.scatter(matrice_Gem[select_row[:40]], matrice_Gem_TL[select_row[:40]], color="tab:blue", label="Gemini")
# plt.title(r'$M^{:} \times E^{:} \times T^{:}$ on Transformer'.format(factors[0], factors[1], factors[2]))
# plt.xlabel('Simulation by Gemini')
# plt.ylabel('Simulating by Timeloop')
# plt.show()


select_row = np.argsort(matrice_TLP)
plt.scatter(matrice_TLP[select_row[:40]]/np.max(matrice_TLP[select_row[:40]]), matrice_TLP_TL[select_row[:40]]/np.max(matrice_TLP_TL[select_row[:40]]), color='tab:orange', label="Predictor")
# plt.title(r'$M^{:} \times E^{:} \times T^{:}$ on Transformer'.format(factors[0], factors[1], factors[2]))
# plt.xlabel('Simulation by Predictor')
# plt.ylabel('Simulating by Timeloop')

plt.xlim([-0.05,1.05])
plt.ylim([-0.05,1.05])
plt.plot([-0.05,1.05],[-0.05,1.05], '--', color='black')
plt.title(r'$M^{:} \times E^{:} \times T^{:}$ on Transformer'.format(factors[0], factors[1], factors[2]))
plt.xlabel('Simulation w/o Timeloop')
plt.ylabel('Simulating w/ Timeloop')
plt.legend()
plt.show()

In [ ]:
data_x_G = data_Gem["cost_overall"].astype(float)
data_y_G = data_Gem["edp"].astype(float)
valid_idx = data_y_G<1e24
data_x_G = data_x_G[valid_idx]
data_y_G = data_y_G[valid_idx]

data_x_TL = data_TL["cost_overall"].astype(float)
data_y_TL = data_TL["edp"].astype(float)
valid_idx = data_y_TL<1e24
data_x_TL = data_x_TL[valid_idx]
data_y_TL = data_y_TL[valid_idx]

medp_G = data_x_G * data_y_G
min_G = medp_G.idxmin()
medp_TL = data_x_TL * data_y_TL
min_TL = medp_TL.idxmin()


plt.scatter(data_x_G, data_y_G, color='tab:blue', label="Simulation by Gemini")
plt.scatter(data_x_TL, data_y_TL, color='tab:orange', label="Simulation by Timeloop")
# plot best medp
plt.scatter(data_x_G[min_G], data_y_G[min_G], marker='D', color='tab:blue', edgecolors='black', label='min MEDP')
plt.scatter(data_x_TL[min_TL], data_y_TL[min_TL], marker='D', color='tab:orange', edgecolors='black', label='min MEDP')
plt.xscale('log')
plt.yscale('log')

plt.minorticks_on()
plt.tick_params(which='minor', length=4, color='gray')
plt.legend(loc='upper right')
plt.xlabel("Manufacturing cost")
plt.ylabel("EDP")
plt.title("Results of Transformer on 128 Tops Chiplet Accelerator")

print("MEDP: {:.3e} --> {:.3e}".format(medp_G[min_G], medp_TL[min_TL]))
print(" EDP: {:.3e} --> {:.3e}".format(data_y_G[min_G], data_y_TL[min_TL]))
print("   M: {:.3e} --> {:.3e}".format(data_x_G[min_G], data_x_TL[min_TL]))

# compare Gem TLP

In [ ]:
# 0: 'mm' ;  1: 'nn' ;  2: 'xx' ;  3: 'yy' ;  4: 'ss' ;  5: 'bb' ;  6: 'rr' ;  7: 'ff' ;  8: 'xcut' ;
# 9: 'ycut' ;  10: 'nop' ;  11: 'ddr' ;  12: 'noc' ;  13: 'mac' ;  14: 'ul3' ;  15: 'tops'

# 0: 'cost_overall' ;  1: 'energy' ;  2: 'cycle' ;  3: 'edp' ;  4: 'cost' ;  5: 'idx' ;  6: 'ubuf' ;  7: 'buf' ;  8: 'bus' ;  
# 9: 'mac.1' ;  10: 'NoC' ;  11: 'NoP' ;  12: 'DRAM' ;  13: 'compute_die_area' ;  14: 'IO_die_area' ;  15: 'os_area' ;  
# 16: 'cost_compute' ;  17: 'cost_IO' ;  18: 'cost_os_overall' ;  19: 'cost_soc'

In [ ]:
data_Gem = pd.read_csv('./GEMINI-32nm/dse_log/eyeriss_resnet50_B64/result.csv').astype(float)
data_TLP = pd.read_csv('./GEMINI-TL/dse_log/eyeriss_resnet50_B64/result.csv').astype(float)

# data_Gem = pd.read_csv('./GEMINI-32nm/dse_log/eyeriss_transformer_B64/result.csv').astype(float)
# data_TLP = pd.read_csv('./GEMINI-TL/dse_log/eyeriss_transformer_B64/result.csv').astype(float)


In [ ]:
mapping_gem = np.zeros(data_Gem.values.shape[0], dtype=int)
for data_idx in range(len(mapping_gem)):
    mapping_rst = data_Gem.values[data_idx,:16] == data_TLP.values[:,:16]
    mapping_rst = np.sum(mapping_rst, axis=1) == 16
    try:
        mapping_gem[data_idx] = np.where(mapping_rst)[0][0]
    except:
        mapping_gem[data_idx] = -1

In [ ]:
data_Gem_val = data_Gem.values[mapping_gem>-1]
data_TLP_val = data_TLP.values[mapping_gem[mapping_gem>-1]]

valid_mapping = data_TLP_val[:,16] < 1e22
data_Gem_val = data_Gem_val[valid_mapping]
data_TLP_val  = data_TLP_val[valid_mapping]

Gimini to TL

In [ ]:
results_index = data_Gem.columns[16:]
# targets = [results_index[0], results_index[1], results_index[2]]
targets = [0, 1, 2]
factors = [1, 1, 2]

data_x, data_y = np.ones(data_Gem_val.shape[0]), np.ones(data_Gem_val.shape[0])
for i in range(len(factors)):
    data_x = data_x * (data_Gem_val[:,targets[i]+16] ** factors[i])
    # data_y = data_y * (data_TLP_val[:,targets[i]+16] ** factors[i])
    data_y = data_y * (data_TLP_val[:,targets[i]+16] ** factors[i])

# axis_min = min(np.min(data_x),np.min(data_y))
# axis_max = max(np.max(data_x),np.max(data_y))
# axis_lim = [ axis_min-0.05*(axis_max-axis_min) , axis_max+0.05*(axis_max-axis_min) ]

select_row = np.argsort(data_x)

# plt.scatter(data_x[select_row[:20]], data_y[select_row[:20]])
plt.scatter(data_x[select_row[:40]], data_y[select_row[:40]])
# plt.plot(axis_lim, axis_lim, '--', color='black')
# plt.legend()
plt.title(r'$M^{:} \times E^{:} \times T^{:}$ on ResNet50'.format(factors[0], factors[1], factors[2]))
plt.xlabel('Simulation w/o Predictor (Gemini)')
plt.ylabel('Simulation w/ Predictor')

# select_col = np.array([0,1,2,3,8,9,10,11,12,13,14])
# print(data_Gemini.columns[select_col])
# print_data = np.array(data_TL.values)[(select_row[:10].values)].astype(int)
# print_data = print_data[:, select_col]
# print_data[:,7] = print_data[:,7] // 1024
# print(print_data)
# print(data_x[select_row[:10]].values / data_y[select_row[:10]].values)

In [ ]:
candidates = data_Gem_val[np.argsort(data_x)[:100],:16]
np.savetxt("eyeriss_resnet50_tops128.log", candidates, fmt="%d", delimiter=" ")

In [ ]:
def is_pareto_efficient(costs, edp):
    is_efficient = np.ones(costs.shape[0], dtype=bool)
    for i, (c, e) in enumerate(zip(costs, edp)):
        if is_efficient[i]:
            # Any point that dominates current point
            dominates = (costs <= c) & (edp <= e) & ((costs < c) | (edp < e))
            is_efficient[dominates] = False
            is_efficient[i] = True
    return is_efficient


total_data_num = 100
use_data = data_TLP
print_name = "TL_transformer"

use_data["edp"] = use_data["energy"] * use_data["cycle"]
pareto_mask = is_pareto_efficient(use_data["cost_overall"].values, use_data["edp"].values)
data_pf = use_data[pareto_mask]

# Define sub-Pareto front: within 10% in both cost and edp of any Pareto point
data_remain = use_data[~pareto_mask]
sub_mask = np.zeros(data_remain["cost_overall"].values.shape[0], dtype=bool)
for _, p in data_pf.iterrows():
    cond = ((data_remain["cost_overall"] <= (p["cost_overall"] * 1.1)) & (data_remain["edp"] <= (p["edp"] * 1.1)))
    sub_mask |= cond
data_spf = data_remain[sub_mask]
data_spf["MEDP"] = data_spf["cost_overall"]*data_spf["edp"]

topk = data_spf.sort_values(by="MEDP").head(total_data_num-data_pf["edp"].values.shape[0])

data_print = np.concatenate([data_pf.values[:,:16], topk.values[:,:16]], axis=0)
np.savetxt(print_name+".log", data_print, fmt="%d", delimiter=" ")

In [ ]:
print(data_pf.shape)

# plot pareto front

In [ ]:
data_Gem = pd.read_csv('./GEMINI-TL-real/dse_log/G_resnet50/result.csv')
data_TLP = pd.read_csv('./GEMINI-TL-real/dse_log/TL_resnet50/result.csv')

In [ ]:
data_x_G = data_Gem["cost_overall"].astype(float)
data_y_G = data_Gem["edp"].astype(float)
valid_idx = data_y_G<1e24
data_x_G = data_x_G[valid_idx]
data_y_G = data_y_G[valid_idx]

data_x_TL = data_TLP["cost_overall"].astype(float)
data_y_TL = data_TLP["edp"].astype(float)
valid_idx = data_y_TL<1e24
data_x_TL = data_x_TL[valid_idx]
data_y_TL = data_y_TL[valid_idx]

medp_G = data_x_G * data_y_G
min_G = medp_G.idxmin()
medp_TL = data_x_TL * data_y_TL
min_TL = medp_TL.idxmin()

# plot all scatter
# plt.scatter(np.log(data_x_G), np.log(data_y_G), color='tab:blue', label="w/o predictor")
# plt.scatter(np.log(data_x_TL), np.log(data_y_TL), color='tab:orange', label="w/ predictor")
# # plot best medp
# plt.scatter(np.log(data_x_G)[min_G], np.log(data_y_G)[min_G], marker='D', color='tab:blue', edgecolors='black', label='min MEDP')
# plt.scatter(np.log(data_x_TL)[min_TL], np.log(data_y_TL)[min_TL], marker='D', color='tab:orange', edgecolors='black', label='min MEDP')
plt.scatter(data_x_G, data_y_G, color='tab:blue', label="w/o predictor")
plt.scatter(data_x_TL, data_y_TL, color='tab:orange', label="w/ predictor")
# plot best medp
plt.scatter(data_x_G[min_G], data_y_G[min_G], marker='D', color='tab:blue', edgecolors='black', label='min MEDP')
plt.scatter(data_x_TL[min_TL], data_y_TL[min_TL], marker='D', color='tab:orange', edgecolors='black', label='min MEDP')
plt.xscale('log')
plt.yscale('log')

plt.minorticks_on()
plt.tick_params(which='minor', length=4, color='gray')
plt.legend(loc='upper right')
plt.xlabel("Manufacturing cost (log10)")
plt.ylabel("EDP (log10)")
plt.title("Results of ResNet50 on 128 Tops Chiplet Accelerator")

print("MEDP: {:.3e} --> {:.3e}".format(medp_G[min_G], medp_TL[min_TL]))
print(" EDP: {:.3e} --> {:.3e}".format(data_y_G[min_G], data_y_TL[min_TL]))
print("   M: {:.3e} --> {:.3e}".format(data_x_G[min_G], data_x_TL[min_TL]))


# analyze seperate log

In [ ]:
# from pyscripts.log_parser import *

import re
import pandas as pd

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def parse_chiplet_log(file_path):
    
    with open(file_path, "r") as file:
        log_lines = file.readlines()

    top_data_all = []

    layer_data_preSPM = []
    sub_layer_data_preSPM = []

    layer_data_postSPM = []
    sub_layer_data_postSPM = []

    Top_pattern = re.compile(
        r'T\s+\d+/\d+\s+E:([\deE+.-]+),\s+T:(\d+),\s+Cost:([\deE+.-]+)\s+'
        r'Ubuf/Buf/Bus/Mac:([\deE+.-]+)/([\deE+.-]+)/([\deE+.-]+)/([\deE+.-]+)\s+'
        r'NoC energy = ([\deE+.-]+)\s+NoP energy = ([\deE+.-]+)\s+DRAM energy = ([\deE+.-]+)\s+'
        r'NoC\(NoC_tot_hops= (\d+)\s+NoP_tot_hops= (\d+),core_hops=(\d+), DRAM acc=(\d+)\)\s+'
        r'NoCtime = (\d+)\s+NocBW = (\d+)'
    )

    LG_pattern = re.compile(
        r'S\s+\d+/\d+\s+E:([\deE+.-]+),\s+T:(\d+),\s+Cost:([\deE+.-]+)\s+'
        r'Ubuf/Buf/Bus/Mac:([\deE+.-]+)/([\deE+.-]+)/([\deE+.-]+)/([\deE+.-]+)\s+'
        r'NoC energy = ([\deE+.-]+)\s+NoP energy = ([\deE+.-]+)\s+DRAM energy = ([\deE+.-]+)\s+'
        r'NoC\(NoC_tot_hops= (\d+)\s+NoP_tot_hops= (\d+),core_hops=(\d+), DRAM acc=(\d+)\)\s+'
        r'NoCtime = (\d+)\s+NocBW = (\d+)'
    )
    sub_LG_pattern = re.compile(
        r'\s*(\w[\w\d_]+)\s+1\s+\(([^)]+)\)\s+util:([\d.]+)/([\d.]+)\s+E:([\deE+.+-]+),\s+T:(\d+),\s+Cost:([\deE+.+-]+)\s+'
        r'Ubuf/Buf/Bus/Mac:([\deE+.+-]+)/([\deE+.+-]+)/([\deE+.+-]+)/([\deE+.+-]+).*?'
        r'NoC energy = ([\deE+.+-]+).*?DRAM energy = ([\deE+.+-]+).*?NoC\(NoC_tot_hops= (\d+)[^,]*,.*?DRAM acc=(\d+).*?'
        r'Computing_time = (\d+)'
    )

    counter = 0
    postSPM = False
    for line in log_lines:
        match_Top    = Top_pattern.search(line)
        match_LG     = LG_pattern.search(line)
        match_sub_LG = sub_LG_pattern.search(line)


        ## get MD cost
        line = line.strip()
        if line.startswith("cost_overall:")

        ## check if is before SPM or not
        if line.startswith("Schtree_spm (after SPM search):"):
            counter = 0
            postSPM = True

        if match_Top:
            Top_data = {
                "energy":       float(match_Top.group(1)),
                "latency":      int(  match_Top.group(2)),
                "cost":         float(match_Top.group(3)),
                "ubuf":         float(match_Top.group(4)),
                "buf":          float(match_Top.group(5)),
                "bus":          float(match_Top.group(6)),
                "mac":          float(match_Top.group(7)),
                "NoC_energy":   float(match_Top.group(8)),
                "NoP_energy":   float(match_Top.group(9)),
                "DRAM_energy":  float(match_Top.group(10)),
                "NoC_tot_hops": int(  match_Top.group(11)),
                "NoP_tot_hops": int(  match_Top.group(12)),
                "core_hops":    int(  match_Top.group(13)),
                "DRAM_acc":     int(  match_Top.group(14)),
                "NoCtime":      int(  match_Top.group(15)),
                "NoC_BW":       int(  match_Top.group(16)),
                "LG_iter":      0
            }
            top_data_all.append(Top_data)

        elif match_LG:
            counter = counter+1
            LG_data = {
                "energy":       float(match_LG.group(1)),
                "latency":      int(  match_LG.group(2)),
                "cost":         float(match_LG.group(3)),
                "ubuf":         float(match_LG.group(4)),
                "buf":          float(match_LG.group(5)),
                "bus":          float(match_LG.group(6)),
                "mac":          float(match_LG.group(7)),
                "NoC_energy":   float(match_LG.group(8)),
                "NoP_energy":   float(match_LG.group(9)),
                "DRAM_energy":  float(match_LG.group(10)),
                "NoC_tot_hops": int(  match_LG.group(11)),
                "NoP_tot_hops": int(  match_LG.group(12)),
                "core_hops":    int(  match_LG.group(13)),
                "DRAM_acc":     int(  match_LG.group(14)),
                "NoCtime":      int(  match_LG.group(15)),
                "NoC_BW":       int(  match_LG.group(16)),
                "LG_iter":      int(  counter)
            }
            if postSPM:
                layer_data_postSPM.append(LG_data)
            else:
                layer_data_preSPM.append(LG_data)

        elif match_sub_LG:
            sub_LG_data = {
                "layer": match_sub_LG.group(1),
                "params": match_sub_LG.group(2),
                "util_actual":      float(match_sub_LG.group(3)),
                "util_peak":        float(match_sub_LG.group(4)),
                "energy":           float(match_sub_LG.group(5)),
                "latency":          int(  match_sub_LG.group(6)),
                "cost":             float(match_sub_LG.group(7)),
                "ubuf":             float(match_sub_LG.group(8)),
                "buf":              float(match_sub_LG.group(9)),
                "bus":              float(match_sub_LG.group(10)),
                "mac":              float(match_sub_LG.group(11)),
                "NoC_energy":       float(match_sub_LG.group(12)),
                "DRAM_energy":      float(match_sub_LG.group(13)),
                "NoC_hops":         int(  match_sub_LG.group(14)),
                "DRAM_acc":         int(  match_sub_LG.group(15)),
                "Computing_time":   int(  match_sub_LG.group(16)),
                "LG_iter":          int(  counter)
            }
            if postSPM:
                sub_layer_data_postSPM.append(sub_LG_data)
            else:
                sub_layer_data_preSPM.append(sub_LG_data)

    preSPM_data = {
        'top':   top_data_all[0], 
        'LG':    pd.DataFrame(layer_data_preSPM),
        'layer': pd.DataFrame(sub_layer_data_preSPM)
    }

    postSPM_data = {
        'top':   top_data_all[1], 
        'LG':    pd.DataFrame(layer_data_postSPM),
        'layer': pd.DataFrame(sub_layer_data_postSPM)
    }

    return preSPM_data, postSPM_data

In [ ]:
preSPM_data, postSPM_data = parse_chiplet_log("./dse_log/resnet50_72tops_Eyeriss_32nm_TL/resnet50/DRAM_bw_294912/NoC_bw_8/ul3_256/NoP_bw_4/mac_num_128/xcut_2_ycut_1/out.log")

In [ ]:
# plot bar chat
plt_data_pre = preSPM_data['top']
plt_data_pos = postSPM_data['top']

